In [65]:
import trimesh
import pyvista as pv
print(pv.__version__) # Check PyVista version
import numpy as np
import matplotlib.pyplot as plt
import os
# VTK imports
from vtkmodules.vtkFiltersSources import vtkLineSource
from vtkmodules.vtkRenderingCore import (
    vtkPolyDataMapper2D,
    vtkActor2D,
    vtkTextActor,
    vtkCoordinate
)
# Assuming your cross-section functions are importable
# from calculate_cross_section_clean import get_section_points_at_plane # Hypothetical function
# from cross_section_functions import order_points
# --- ICP Alignment (Optional but Recommended) ---
try:
    from trimesh.registration import icp
    use_icp = True
except ImportError:
    print("Warning: 'open3d' or 'pycpd' not found. ICP alignment will be skipped.")
    use_icp = False
# --- End ICP ---


def compare_meshes(original_obj_path, idealised_ply_path, output_dir, base_name):
    """Generates comparison figures for original and idealised meshes, displayed side-by-side."""

    print(f"Comparing '{original_obj_path}' and '{idealised_ply_path}'")
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    try:
        # 1. Load Meshes
        original_mesh = trimesh.load_mesh(original_obj_path)
        idealised_mesh = trimesh.load_mesh(idealised_ply_path)

        if isinstance(original_mesh, trimesh.Scene):
            original_mesh = original_mesh.dump(concatenate=True)
        if isinstance(idealised_mesh, trimesh.Scene):
            idealised_mesh = idealised_mesh.dump(concatenate=True)

        # --- Mesh Integrity Checks ---
        print(f"Original Mesh - Watertight: {original_mesh.is_watertight}")
        print(f"Idealised Mesh - Watertight: {idealised_mesh.is_watertight}")
        # Optional: Try processing to fix minor issues
        # original_mesh.process()
        # idealised_mesh.process()
        # --- End Checks ---

        # 2. Align Meshes
        # Start with centroid alignment
        center_orig = original_mesh.centroid
        center_ideal = idealised_mesh.centroid
        translation = center_orig - center_ideal
        if np.linalg.norm(translation) > 1e-4: # Only translate if significantly different
            print(f"  Aligning idealised mesh centroid to original ({translation})...")
            idealised_mesh.apply_translation(translation)

        # --- Advanced Alignment using ICP (Optional) ---
        if use_icp:
            try:
                # Sample points from both meshes for efficiency
                points_orig, _ = trimesh.sample.sample_surface(original_mesh, 2000)
                points_ideal, _ = trimesh.sample.sample_surface(idealised_mesh, 2000)

                # Run ICP
                icp_result = icp(points_ideal, points_orig, scale=False, max_iterations=100)

                transform_icp = None
                cost = None

                # Check the number of returned values and unpack carefully
                if isinstance(icp_result, (list, tuple)):
                    if len(icp_result) == 3:
                        transform_icp, cost, _ = icp_result
                    elif len(icp_result) == 2:
                        transform_icp, cost = icp_result
                    elif len(icp_result) == 1: # Maybe only transform is returned
                         transform_icp = icp_result[0]
                         cost = -1.0 # Indicate unknown cost
                    else:
                        print("  Warning: ICP returned an unexpected number of values.")
                        cost = -2.0 # Indicate error
                elif isinstance(icp_result, np.ndarray) and icp_result.shape == (4, 4):
                    # Handle case where only the transform matrix is returned
                    print("  Warning: ICP might have returned only the transformation matrix.")
                    transform_icp = icp_result
                    cost = -1.0 # Indicate unknown cost
                else:
                     print("  Warning: ICP returned an unknown format.")
                     cost = -3.0 # Indicate error

                # Validate transform_icp before proceeding
                if transform_icp is None or not isinstance(transform_icp, np.ndarray) or transform_icp.shape != (4, 4):
                    raise ValueError("Failed to retrieve a valid 4x4 transformation matrix from ICP.")

                # Process cost: Ensure it's a float before printing/using
                final_cost_str = "N/A"
                if cost is not None:
                    try:
                        # Attempt to convert cost to float, handle arrays
                        if isinstance(cost, np.ndarray):
                            if cost.size == 1:
                                cost_float = float(cost.item())
                            else:
                                # If cost is an array with multiple values, maybe take the mean or first?
                                print(f"  Warning: ICP cost is an array ({cost}), using first element.")
                                cost_float = float(cost.flat[0])
                        else:
                            cost_float = float(cost)

                        if np.isfinite(cost_float):
                             final_cost_str = f"{cost_float:.6f}"
                        else:
                             final_cost_str = "Invalid (non-finite)"
                    except (TypeError, ValueError) as cost_err:
                        print(f"  Warning: Could not interpret ICP cost value ({cost}): {cost_err}")
                        final_cost_str = f"Error ({cost})"

                print(f"  ICP alignment cost: {final_cost_str}")

                # Apply the found transformation to the idealised mesh
                idealised_mesh.apply_transform(transform_icp)
                print("  Applied ICP transformation to idealised mesh.")

            except Exception as e:
                # Print traceback for unexpected errors during ICP itself
                import traceback
                print(f"  Warning: Error during ICP alignment: {e}.")
                # traceback.print_exc() # Uncomment for full traceback if needed
                print("  Proceeding without ICP.")
        # --- End ICP Alignment ---


        # Convert to PyVista objects AFTER alignment
        original_pv = pv.wrap(original_mesh)
        idealised_pv = pv.wrap(idealised_mesh)

        # --- Calculate combined bounds AFTER alignment for ruler placement ---
        combined_bounds = np.array(original_pv.bounds)
        combined_bounds[0] = min(combined_bounds[0], idealised_pv.bounds[0])
        combined_bounds[1] = max(combined_bounds[1], idealised_pv.bounds[1])
        combined_bounds[2] = min(combined_bounds[2], idealised_pv.bounds[2])
        combined_bounds[3] = max(combined_bounds[3], idealised_pv.bounds[3])
        combined_bounds[4] = min(combined_bounds[4], idealised_pv.bounds[4])
        combined_bounds[5] = max(combined_bounds[5], idealised_pv.bounds[5])

        center_x = (combined_bounds[0] + combined_bounds[1]) / 2.0
        center_y = (combined_bounds[2] + combined_bounds[3]) / 2.0
        center_z = (combined_bounds[4] + combined_bounds[5]) / 2.0
        bottom_y = combined_bounds[2] # Y min
        bottom_z = combined_bounds[4] # Z min

        views = {'xy': 'top', 'xz': 'side', 'yz': 'front'}
        original_color = 'grey'
        idealised_color = '#D55E00'

        for view_plane, view_name in views.items():
            plotter = pv.Plotter(shape=(1, 2), off_screen=True, window_size=[2048, 1024], border=False)

            ruler_length_world = 5.0
            ruler_screen_offset_factor = 0.9 # Current value
            ruler_depth_offset_factor = 0.25  # Current value

            xy_text_position = (0.47, 0.05)
            xz_text_position = (0.47, 0.31)
            yz_text_position = (0.47, 0.09)

            current_text_position = None
            if view_plane == 'xy':
                current_text_position = xy_text_position
            elif view_plane == 'xz':
                current_text_position = xz_text_position
            elif view_plane == 'yz':
                current_text_position = yz_text_position

            p0_ruler, p1_ruler = None, None
            if view_plane == 'xy':
                ruler_y_world = bottom_y - (ruler_length_world * ruler_screen_offset_factor)
                ruler_z_world = combined_bounds[5] + (ruler_length_world * ruler_depth_offset_factor)
                p0_ruler = np.array([center_x - ruler_length_world / 2.0, ruler_y_world, ruler_z_world])
                p1_ruler = np.array([center_x + ruler_length_world / 2.0, ruler_y_world, ruler_z_world])
            elif view_plane == 'xz':
                ruler_z_world = bottom_z - (ruler_length_world * ruler_screen_offset_factor)
                ruler_y_world = center_y
                p0_ruler = np.array([center_x - ruler_length_world / 2.0, ruler_y_world, ruler_z_world])
                p1_ruler = np.array([center_x + ruler_length_world / 2.0, ruler_y_world, ruler_z_world])
            elif view_plane == 'yz':
                ruler_y_world = bottom_y - (ruler_length_world * ruler_screen_offset_factor)
                ruler_x_world = combined_bounds[0] - (ruler_length_world * ruler_depth_offset_factor)
                p0_ruler = np.array([ruler_x_world, ruler_y_world, center_z - ruler_length_world / 2.0])
                p1_ruler = np.array([ruler_x_world, ruler_y_world, center_z + ruler_length_world / 2.0])

            # Calculate bounds for the ruler itself
            ruler_bounds = np.array([
                min(p0_ruler[0], p1_ruler[0]), max(p0_ruler[0], p1_ruler[0]),
                min(p0_ruler[1], p1_ruler[1]), max(p0_ruler[1], p1_ruler[1]),
                min(p0_ruler[2], p1_ruler[2]), max(p0_ruler[2], p1_ruler[2])
            ])

            # Create overall_bounds that encompass both meshes and the ruler
            overall_bounds = np.copy(combined_bounds)
            overall_bounds[0] = min(overall_bounds[0], ruler_bounds[0]) # min_x
            overall_bounds[1] = max(overall_bounds[1], ruler_bounds[1]) # max_x
            overall_bounds[2] = min(overall_bounds[2], ruler_bounds[2]) # min_y
            overall_bounds[3] = max(overall_bounds[3], ruler_bounds[3]) # max_y
            overall_bounds[4] = min(overall_bounds[4], ruler_bounds[4]) # min_z
            overall_bounds[5] = max(overall_bounds[5], ruler_bounds[5]) # max_z

            # --- Subplot 1 ---
            plotter.subplot(0, 0)
            plotter.add_mesh(original_pv, color=original_color, smooth_shading=False, show_edges=False)

            if view_plane == 'xy':
                plotter.view_xy()
                plotter.reset_camera(bounds=overall_bounds) # Use OVERALL bounds
            elif view_plane == 'xz':
                plotter.view_xz()
                plotter.reset_camera(bounds=overall_bounds) # Use OVERALL bounds
            elif view_plane == 'yz':
                plotter.view_yz()
                plotter.camera.position = [center_x, center_y, center_z]
                plotter.camera.focal_point = [center_x - 1, center_y, center_z]
                plotter.camera.up = (0, 1, 0)
                plotter.reset_camera(bounds=overall_bounds) # Use OVERALL bounds
            plotter.camera.zoom(1.2)

            if p0_ruler is not None and p1_ruler is not None:
                line_actor_s1 = pv.Line(p0_ruler, p1_ruler)
                plotter.add_mesh(line_actor_s1, color='black', line_width=10, pickable=False, culling=False)
            if current_text_position:
                plotter.add_text("5 µm", position=current_text_position, font_size=16, color='black', font='arial', viewport=True, shadow=True)

            # --- Subplot 2 ---
            plotter.subplot(0, 1)
            plotter.add_mesh(idealised_pv, color=idealised_color, smooth_shading=False, show_edges=False)

            if view_plane == 'xy':
                plotter.view_xy()
                plotter.reset_camera(bounds=overall_bounds) # Use OVERALL bounds
            elif view_plane == 'xz':
                plotter.view_xz()
                plotter.reset_camera(bounds=overall_bounds) # Use OVERALL bounds
            elif view_plane == 'yz':
                plotter.view_yz()
                plotter.camera.position = [center_x, center_y, center_z]
                plotter.camera.focal_point = [center_x - 1, center_y, center_z]
                plotter.camera.up = (0, 1, 0)
                plotter.reset_camera(bounds=overall_bounds) # Use OVERALL bounds
            plotter.camera.zoom(1.2)

            if p0_ruler is not None and p1_ruler is not None:
                line_actor_s2 = pv.Line(p0_ruler, p1_ruler)
                plotter.add_mesh(line_actor_s2, color='black', line_width=10, pickable=False, culling=False)
            if current_text_position:
                plotter.add_text("5 µm", position=current_text_position, font_size=16, color='black', font='arial', viewport=True, shadow=True)

            plotter.link_views()

            # --- Screenshot ---
            output_path = os.path.join(output_dir, f"{base_name}_comparison_{view_name}_sidebyside.png")
            print(f"Saving image to: {output_path}")
            plotter.screenshot(output_path, transparent_background=True)
            plotter.close()

    except Exception as e:
        print(f"  Error processing comparison: {e}")
        import traceback
        traceback.print_exc()

# --- Example Usage ---
# Make sure this cell runs if you intend to execute the comparison
# (In Jupyter, you might run this cell directly)
# if __name__ == '__main__': # This check might not be necessary in a notebook cell
original_file = "Meshes/OBJ/Ac_DA_1_3.obj" # Example
idealised_file_std = "results/full_stomata_Ac_DA_1_3_std.ply" # Example
idealised_file_bulge = "results/full_stomata_Ac_DA_1_3_bulge.ply" # Example
comparison_output_dir = "results/comparisons"

base_name_orig = os.path.splitext(os.path.basename(original_file))[0]

if os.path.exists(original_file) and os.path.exists(idealised_file_std):
    compare_meshes(original_file, idealised_file_std, comparison_output_dir, f"{base_name_orig}_vs_std")

if os.path.exists(original_file) and os.path.exists(idealised_file_bulge):
    compare_meshes(original_file, idealised_file_bulge, comparison_output_dir, f"{base_name_orig}_vs_bulge")


0.44.2
Comparing 'Meshes/OBJ/Ac_DA_1_3.obj' and 'results/full_stomata_Ac_DA_1_3_std.ply'
Original Mesh - Watertight: False
Idealised Mesh - Watertight: False
  Aligning idealised mesh centroid to original ([-2.27768049e-02  5.57586929e-02 -4.07465878e-08])...
 [  1.28202404   7.15933207   1.80681678]
 [ 11.35852011   9.02573337  -7.01409921]
 ...
 [  1.01518226  -8.79165731  -0.63452264]
 [  9.96122674  -5.82734353   4.89954541]
 [ -3.52638473  10.99034109   5.40519084]]), using first element.
  ICP alignment cost: 9.033799
  Applied ICP transformation to idealised mesh.
Saving image to: results/comparisons/Ac_DA_1_3_vs_std_comparison_top_sidebyside.png
Saving image to: results/comparisons/Ac_DA_1_3_vs_std_comparison_side_sidebyside.png
Saving image to: results/comparisons/Ac_DA_1_3_vs_std_comparison_front_sidebyside.png
Comparing 'Meshes/OBJ/Ac_DA_1_3.obj' and 'results/full_stomata_Ac_DA_1_3_bulge.ply'
Original Mesh - Watertight: False
Idealised Mesh - Watertight: False
  Aligning ide

In [ ]:
import trimesh
import pyvista as pv
import numpy as np
import os
from sklearn.decomposition import PCA
import math
# Ensure cross_section_functions is importable, e.g., if it's cross_section_functions.py
# in the same directory or installed.
try:
    import cross_section_functions as csf
except ImportError:
    print("Error: cross_section_functions.py (csf) not found. Please ensure it's accessible.")
    # You might need to add to sys.path or ensure it's a proper module
    # For example:
    # import sys
    # sys.path.append(os.path.abspath('.')) # If in the same directory
    # import cross_section_functions as csf
    # As a fallback, create a dummy csf if not found, so the rest of the notebook can be partially tested
    # This is NOT a real solution for csf, just to prevent NameErrors for csf.order_points etc.
    class DummyCSF:
        def load_and_align_mesh(self, mesh_path, align_axis='Y'):
            print(f"DummyCSF: load_and_align_mesh called for {mesh_path}")
            try:
                mesh = trimesh.load_mesh(mesh_path)
                if isinstance(mesh, trimesh.Scene):
                    mesh = mesh.dump(concatenate=True)
                # Basic alignment: center and orient along Y by inertia tensor (simplified)
                mesh.vertices -= mesh.centroid
                inertia = mesh.moment_inertia_tensor
                eigvals, eigvecs = np.linalg.eigh(inertia)
                longest_axis_vec = eigvecs[:, np.argmax(eigvals)]
                target_axis_vec = np.array([0,1,0] if align_axis == 'Y' else [1,0,0])
                rotation = trimesh.transformations.rotation_matrix(
                    np.arccos(np.dot(longest_axis_vec, target_axis_vec)),
                    np.cross(longest_axis_vec, target_axis_vec)
                )[0:3, 0:3]
                mesh.apply_transform(rotation)
                return mesh, None # No transform matrix returned by dummy
            except Exception as e:
                print(f"DummyCSF: Error in load_and_align_mesh: {e}")
                return None, None
        
        def get_radial_dimensions(self, aligned_mesh, center, ray_count):
            print(f"DummyCSF: get_radial_dimensions called.")
            # Return plausible dummy data
            # This is highly dependent on mesh structure and won't be accurate
            # For testing, we need some centerline points.
            # Let's assume a simple straight centerline along Y.
            min_y, max_y = aligned_mesh.bounds[:,1]
            raw_centerline_points = np.array([[0, y, 0] for y in np.linspace(min_y, max_y, 10)])
            minor_radius = np.mean(aligned_mesh.extents) / 4 # very rough guess
            return None, None, raw_centerline_points, minor_radius

        def order_points(self, points, method="angular"):
            print(f"DummyCSF: order_points called with {len(points)} points.")
            if len(points) == 0: return np.array([])
            # Simplified angular sort around centroid
            centroid = np.mean(points, axis=0)
            angles = np.arctan2(points[:,1] - centroid[1], points[:,0] - centroid[0])
            return points[np.argsort(angles)]

        # filter_section_points is the one we are replacing, so no dummy needed for it here.

    if 'csf' not in globals(): # If import failed
        csf = DummyCSF()
        print("WARNING: Using DummyCSF as cross_section_functions.py was not found. Results will be incorrect.")


def get_midpoint_section_2d_points(mesh_path):
    """
    Processes a single mesh file to extract its 2D midpoint cross-section (through the wall),
    oriented horizontally for 2D plots. The returned 3D section vertices will correspond to the
    actual cut on the mesh.
    Returns:
        - ordered_points_2d_for_plot_orientation: Horizontally oriented 2D points for 2D plotting.
        - aligned_mesh: The trimesh object of the mesh after alignment.
        - plane_origin_3d: The 3D origin of the cutting plane.
        - plane_normal_3d: The 3D normal of the cutting plane.
        - section_vertices_3d_actual_cut: Ordered 3D vertices of the actual cross-section path on the mesh.
    """
    print(f"    Extracting 2D midpoint section for: {mesh_path}")
    aligned_mesh, _ = csf.load_and_align_mesh(mesh_path, align_axis='Y')
    if aligned_mesh is None:
        print(f"      Failed to load/align mesh: {mesh_path}")
        return None, None, None, None, None

    # Call get_radial_dimensions primarily for minor_radius or if other outputs are needed.
    # The raw_centerline_points from here are NOT used for the plane definition below.
    # ray_origin_for_radial_dims = np.array([0.0, 0.0, 0.0])
    # _, _, _, minor_radius = csf.get_radial_dimensions(
    #     aligned_mesh, center=ray_origin_for_radial_dims, ray_count=36
    # )
    # If minor_radius is not used later by the caller, this whole block could be optional.

    # --- Define plane for a transverse midpoint cut (assuming Y is length axis) ---
    plane_origin_3d = np.array([0.0, 0.0, 0.0]) # At the Y-midpoint of the aligned mesh
    plane_normal_3d = np.array([0.0, 1.0, 0.0]) # Normal along the Y-axis
    print(f"      For midpoint section, using TRANSVERSE plane_origin_3d: {plane_origin_3d.round(3)} and plane_normal_3d: {plane_normal_3d.round(3)}")
    # --- End plane definition ---

    section_3D = aligned_mesh.section(plane_origin=plane_origin_3d, plane_normal=plane_normal_3d)

    if section_3D is None or len(section_3D.entities) == 0:
        print(f"      Midpoint section failed or is empty for: {mesh_path}")
        return None, aligned_mesh, plane_origin_3d, plane_normal_3d, None

    path_2D, transform_2d_to_3d = section_3D.to_2D()
    
    # --- MODIFICATION START (from previous changes, ensure it's robust) ---
    raw_polygons_data = path_2D.polygons_closed
    processed_polygons_list = []
    if raw_polygons_data is not None:
        if isinstance(raw_polygons_data, list):
            for item in raw_polygons_data:
                if hasattr(item, 'vertices') and isinstance(item.vertices, np.ndarray):
                    vertices = item.vertices
                    if vertices.ndim == 2 and vertices.shape[1] == 2 and vertices.shape[0] >= 3:
                        processed_polygons_list.append(vertices)
                elif isinstance(item, np.ndarray): # Handle if list contains ndarrays
                    vertices = item
                    if vertices.ndim == 2 and vertices.shape[1] == 2 and vertices.shape[0] >= 3:
                        processed_polygons_list.append(vertices)
        elif isinstance(raw_polygons_data, np.ndarray):
            if raw_polygons_data.ndim == 2 and raw_polygons_data.shape[1] == 2 and raw_polygons_data.shape[0] >= 3:
                processed_polygons_list.append(raw_polygons_data)
            elif raw_polygons_data.ndim > 0 and len(raw_polygons_data) > 0: # e.g. list of arrays returned as object array
                try:
                    for poly_array in raw_polygons_data:
                        if isinstance(poly_array, np.ndarray) and poly_array.ndim == 2 and poly_array.shape[1] == 2 and poly_array.shape[0] >= 3:
                           processed_polygons_list.append(poly_array)
                except TypeError:
                     print(f"      Warning: polygons_closed (ndarray) has unexpected structure. Mesh: {mesh_path}")
    
    if not processed_polygons_list:
        if hasattr(path_2D, 'discrete') and path_2D.discrete:
            valid_discrete_paths = [p for p in path_2D.discrete if isinstance(p, np.ndarray) and p.ndim == 2 and p.shape[1] == 2 and p.shape[0] >= 3]
            if valid_discrete_paths:
                processed_polygons_list.append(max(valid_discrete_paths, key=len))
                print(f"      Fallback: Used longest discrete path for {mesh_path}.")
            else:
                print(f"      No valid discrete paths found for fallback for: {mesh_path}")
        else:
            print(f"      No closed polygons or discrete paths found for: {mesh_path}")
        if not processed_polygons_list:
            return None, aligned_mesh, plane_origin_3d, plane_normal_3d, None

    final_points_2D_raw_selected_polygon = max(processed_polygons_list, key=len)
    print(f"      Selected polygon with {len(final_points_2D_raw_selected_polygon)} vertices from {len(processed_polygons_list)} polygon(s)/path(s) in the 2D section.")

    if not isinstance(final_points_2D_raw_selected_polygon, np.ndarray):
        final_points_2D_for_ordering = np.array(final_points_2D_raw_selected_polygon)
    else:
        final_points_2D_for_ordering = final_points_2D_raw_selected_polygon
    # --- MODIFICATION END ---

    if len(final_points_2D_for_ordering) < 3:
        print(f"      Not enough points in selected polygon for: {mesh_path} (found {len(final_points_2D_for_ordering)}).")
        return None, aligned_mesh, plane_origin_3d, plane_normal_3d, None

    # These are the 2D points of the actual section, ordered, but not yet PCA-rotated for 2D plot aesthetics
    ordered_points_2d_actual_section = csf.order_points(final_points_2D_for_ordering, method="angular")
    if ordered_points_2d_actual_section is None or len(ordered_points_2d_actual_section) < 3:
        print(f"      Failed to order actual section points for: {mesh_path}")
        return None, aligned_mesh, plane_origin_3d, plane_normal_3d, None

    # Create PCA-rotated 2D points for consistent horizontal orientation in 2D plots
    ordered_points_2d_for_plot_orientation = ordered_points_2d_actual_section # Default to actual section if PCA fails
    if len(ordered_points_2d_actual_section) >= 2: # PCA needs at least 2 points
        try:
            pca = PCA(n_components=2)
            pca.fit(ordered_points_2d_actual_section)
            principal_axis = pca.components_[0] # Longest axis of the 2D shape
            angle_rad = np.arctan2(principal_axis[1], principal_axis[0]) # Angle of this axis
            
            # Rotate to make this principal_axis horizontal
            cos_a = np.cos(-angle_rad)
            sin_a = np.sin(-angle_rad)
            rotation_matrix_2d = np.array([[cos_a, -sin_a], [sin_a, cos_a]])
            
            center_2d = np.mean(ordered_points_2d_actual_section, axis=0)
            centered_points_2d = ordered_points_2d_actual_section - center_2d
            rotated_centered_points_2d = centered_points_2d @ rotation_matrix_2d.T
            ordered_points_2d_for_plot_orientation = rotated_centered_points_2d + center_2d
            print(f"      Oriented 2D section horizontally for 2D plots (angle: {-np.degrees(angle_rad):.1f}°).")
        except Exception as e:
            print(f"      Warning: PCA-based orientation for 2D plots failed: {e}")
            # ordered_points_2d_for_plot_orientation remains ordered_points_2d_actual_section
    
    # Generate 3D section vertices from the ACTUAL (un-PCA-rotated for 2D plot) section points
    section_vertices_3d_actual_cut = None
    if ordered_points_2d_actual_section is not None and len(ordered_points_2d_actual_section) > 0:
        if transform_2d_to_3d is not None:
            # Ensure ordered_points_2d_actual_section is (N,2)
            current_2d_points_for_3d = ordered_points_2d_actual_section
            if current_2d_points_for_3d.ndim == 1:
                 current_2d_points_for_3d = current_2d_points_for_3d.reshape(-1, 2)

            # Add z=0 coordinate for 2D points before transforming to 3D
            points_in_2d_plane_for_transform = np.column_stack((current_2d_points_for_3d, np.zeros(len(current_2d_points_for_3d))))
            section_vertices_3d_actual_cut = trimesh.transform_points(points_in_2d_plane_for_transform, transform_2d_to_3d)
        else:
            # Fallback if transform_2d_to_3d is None (e.g., section.to_2D() failed to produce it)
            print(f"      Warning: transform_2d_to_3d is None for {mesh_path}. Attempting to use 3D polygons/paths directly from section_3D.")
            # These points are already 3D and on the mesh surface.
            # Try to get ordered 3D polygons first
            if hasattr(section_3D, 'polygons_closed') and section_3D.polygons_closed:
                polygons_3d_list = [p for p in section_3D.polygons_closed if isinstance(p, np.ndarray) and p.ndim == 2 and p.shape[0] >=3]
                if polygons_3d_list:
                    section_vertices_3d_actual_cut = max(polygons_3d_list, key=len)
                    print(f"      Using largest 3D polygon from section_3D.polygons_closed (length {len(section_vertices_3d_actual_cut)}) as fallback for 3D section vertices.")
            # If no 3D polygons, try discrete 3D paths
            if section_vertices_3d_actual_cut is None and hasattr(section_3D, 'discrete') and section_3D.discrete:
                 polylines_3d_list = [p for p in section_3D.discrete if isinstance(p, np.ndarray) and p.ndim == 2 and p.shape[0] >=3]
                 if polylines_3d_list:
                    section_vertices_3d_actual_cut = max(polylines_3d_list, key=len)
                    print(f"      Using largest discrete 3D polyline from section_3D.discrete (length {len(section_vertices_3d_actual_cut)}) as fallback for 3D section vertices.")
            
            if section_vertices_3d_actual_cut is None:
                 print(f"      No suitable 3D path fallback found in section_3D when transform_2d_to_3d is None.")

    # Final checks and return
    if ordered_points_2d_for_plot_orientation is None or len(ordered_points_2d_for_plot_orientation) == 0:
        print(f"      Final ordered_points_2d_for_plot_orientation is empty for: {mesh_path}")
        return None, aligned_mesh, plane_origin_3d, plane_normal_3d, None

    if section_vertices_3d_actual_cut is None or len(section_vertices_3d_actual_cut) == 0:
        print(f"      Final section_vertices_3d_actual_cut is empty for: {mesh_path}")
        # Return the 2D points for plotting if 3D path failed but 2D succeeded
        return ordered_points_2d_for_plot_orientation, aligned_mesh, plane_origin_3d, plane_normal_3d, None

    print(f"      Successfully extracted section for: {mesh_path}. 2D points for plot: {len(ordered_points_2d_for_plot_orientation)}, 3D path vertices: {len(section_vertices_3d_actual_cut)}.")
    return ordered_points_2d_for_plot_orientation, aligned_mesh, plane_origin_3d, plane_normal_3d, section_vertices_3d_actual_cut


def plot_3d_section_context(trimesh_mesh_aligned, plane_origin, plane_normal, section_path_vertices_3d, output_filename, mesh_color='lightgrey', section_color='red', plane_color='blue'): # plane_color is effectively unused now for the isometric view
    """
    Generates a 3D visualization of the mesh with its cutting plane and section path.
    Also generates a top-down view showing the section position.
    """
    if trimesh_mesh_aligned is None:
        print(f"  Skipping 3D context plot for {output_filename}, mesh is None.")
        return

    pv_mesh = pv.wrap(trimesh_mesh_aligned)
    # bounds = trimesh_mesh_aligned.bounds # Not strictly needed if plane actor is removed
    # diag_length = np.linalg.norm(bounds[1] - bounds[0]) # Not strictly needed if plane actor is removed

    # --- Plot 1: Isometric 3D Context View ---
    plotter_iso = pv.Plotter(off_screen=True, window_size=[1280, 960], border=False)
    plotter_iso.add_mesh(pv_mesh, color=mesh_color, opacity=0.7, smooth_shading=False, show_edges=False)

    if plane_origin is not None and plane_normal is not None:
        # The explicit cutting_plane actor (square outline) is removed from this view.
        # The section_path_vertices_3d will show where the cut was made.

        if section_path_vertices_3d is not None and len(section_path_vertices_3d) > 1:
            closed_section_path_3d = np.vstack([section_path_vertices_3d, section_path_vertices_3d[0]])
            plotter_iso.add_lines(closed_section_path_3d, color=section_color, width=10, connected=True)

        plotter_iso.view_isometric()
        plotter_iso.enable_parallel_projection()
        plotter_iso.reset_camera()
        plotter_iso.camera.zoom(1.5)
    else:
        plotter_iso.view_isometric()
        plotter_iso.enable_parallel_projection()
        plotter_iso.reset_camera()
        plotter_iso.camera.zoom(1.3)

    print(f"  Saving 3D isometric section context view to: {output_filename}")
    plotter_iso.screenshot(output_filename, transparent_background=True)
    plotter_iso.close()

    # --- Plot 2: Top-Down View for Section Position (Doughnut View) ---
    # This view assumes the mesh has been aligned so that its length is along the Y-axis.
    # view_xy() looks down the Z-axis, which should show the pore if oriented conventionally.
    if plane_origin is not None and plane_normal is not None and section_path_vertices_3d is not None and len(section_path_vertices_3d) > 1:
        plotter_top = pv.Plotter(off_screen=True, window_size=[1024, 1024], border=False)
        plotter_top.add_mesh(pv_mesh, color=mesh_color, opacity=0.85, smooth_shading=False, show_edges=False)
        
        closed_section_path_3d_top = np.vstack([section_path_vertices_3d, section_path_vertices_3d[0]])
        plotter_top.add_lines(closed_section_path_3d_top, color=section_color, width=10, connected=True)

        plotter_top.view_xy() # Looks down the Z-axis (top-down view of the XY plane)
        plotter_top.enable_parallel_projection()
        plotter_top.reset_camera() 
        plotter_top.camera.zoom(1.4) 

        base, ext = os.path.splitext(output_filename)
        top_view_output_filename = f"{base}_top_view{ext}"
        print(f"  Saving 3D top-down section context view to: {top_view_output_filename}")
        plotter_top.screenshot(top_view_output_filename, transparent_background=True)
        plotter_top.close()
    elif plane_origin is None or plane_normal is None:
        print(f"  Skipping top-down view for {output_filename} as plane information is missing.")
    elif section_path_vertices_3d is None or len(section_path_vertices_3d) <= 1:
        print(f"  Skipping top-down view for {output_filename} as section path is missing or too short.")


def plot_midpoint_cross_sections_superimposed(original_mesh_path, idealised_mesh_path, output_dir, base_name):
    """
    Generates a superimposed plot of the 2D midpoint cross-section outlines of two meshes.
    """
    print(f"\nPlotting superimposed midpoint cross-section outlines for '{base_name}'")
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    points_orig_2d, _, _, _, _ = get_midpoint_section_2d_points(original_mesh_path)
    points_ideal_2d, _, _, _, _ = get_midpoint_section_2d_points(idealised_mesh_path)

    if points_orig_2d is None and points_ideal_2d is None:
        print("  Both mesh cross-sections failed. Skipping superimposed plot.")
        return

    plotter = pv.Plotter(off_screen=True, window_size=[1024, 1024], border=False) # Single plot

    original_color = 'grey' # Color for original
    idealised_color = '#D55E00' # Color for idealised (Orange/Red)
    line_width = 5
    ruler_length_world = 5.0
    text_viewport_pos = (0.47, 0.05) # Adjust as needed for single plot

    all_plot_points = []
    if points_orig_2d is not None and len(points_orig_2d) > 0:
        all_plot_points.extend(points_orig_2d)
    if points_ideal_2d is not None and len(points_ideal_2d) > 0:
        all_plot_points.extend(points_ideal_2d)

    global_min_x, global_max_x, global_min_y, global_max_y = (np.inf, -np.inf, np.inf, -np.inf)
    has_data_for_bounds = False

    if all_plot_points:
        all_points_np = np.array(all_plot_points)
        if all_points_np.size > 0 and all_points_np.ndim == 2 and all_points_np.shape[1] == 2:
            global_min_x = np.min(all_points_np[:,0])
            global_max_x = np.max(all_points_np[:,0])
            global_min_y = np.min(all_points_np[:,1])
            global_max_y = np.max(all_points_np[:,1])
            has_data_for_bounds = True
        else:
            print("     Warning: all_plot_points is not in the expected format for bounds calculation (superimposed).")


    padding_for_ruler_area_y = ruler_length_world * 0.4

    if has_data_for_bounds:
        global_min_y_padded = global_min_y - padding_for_ruler_area_y
        min_extent = ruler_length_world * 0.5
        if (global_max_x - global_min_x) < min_extent:
            center_x = (global_min_x + global_max_x) / 2
            global_min_x, global_max_x = center_x - min_extent / 2, center_x + min_extent / 2
        if (global_max_y - global_min_y) < min_extent:
            center_y = (global_min_y + global_max_y) / 2
            global_min_y, global_max_y = center_y - min_extent / 2, center_y + min_extent / 2
            global_min_y_padded = global_min_y - padding_for_ruler_area_y
    else:
        global_min_x, global_max_x = -ruler_length_world, ruler_length_world
        global_min_y_padded = -ruler_length_world * 0.7
        global_max_y = ruler_length_world * 0.3

    legend_entries = []
    if points_orig_2d is not None and len(points_orig_2d) > 1:
        points_orig_3d = np.hstack((points_orig_2d, np.zeros((points_orig_2d.shape[0], 1))))
        closed_points_orig = np.vstack([points_orig_3d, points_orig_3d[0]])
        plotter.add_lines(closed_points_orig, color=original_color, width=line_width, connected=True)        
        legend_entries.append(['Original', original_color])

    if points_ideal_2d is not None and len(points_ideal_2d) > 1:
        points_ideal_3d = np.hstack((points_ideal_2d, np.zeros((points_ideal_2d.shape[0], 1))))
        closed_points_ideal = np.vstack([points_ideal_3d, points_ideal_3d[0]])
        plotter.add_lines(closed_points_ideal, color=original_color, width=line_width, connected=True)        
        legend_entries.append(['Idealised', idealised_color])

    if has_data_for_bounds or legend_entries: # Add ruler if there's any data
        ruler_center_x = (global_min_x + global_max_x) / 2.0
        ruler_y_pos = global_min_y_padded + (padding_for_ruler_area_y * 0.25)
        p0_ruler = np.array([ruler_center_x - ruler_length_world / 2.0, ruler_y_pos, 0])
        p1_ruler = np.array([ruler_center_x + ruler_length_world / 2.0, ruler_y_pos, 0])
        plotter.add_mesh(pv.Line(p0_ruler, p1_ruler), color='black', line_width=5)
        plotter.add_text("5 µm", position=text_viewport_pos, font_size=16, color='black', font='arial', viewport=True, shadow=True)

    if legend_entries:
        plotter.add_legend(labels=legend_entries, bcolor=None, face=None, size=(0.15, 0.15))


    plotter.view_xy()
    plotter.enable_parallel_projection()
    plotter.camera.bounds = [global_min_x, global_max_x, global_min_y_padded, global_max_y, -1, 1]
    plotter.reset_camera()
    #plotter.camera.zoom('auto')

    output_filename = f"{base_name}_midpoint_cs_superimposed.png"
    output_path = os.path.join(output_dir, output_filename)
    print(f"  Saving superimposed 2D cross-section to: {output_path}")
    plotter.screenshot(output_path, transparent_background=True)
    plotter.close()


# --- Example Usage ---
original_file = "Meshes/OBJ/Ac_DA_1_3.obj"
idealised_file_std = "results/full_stomata_Ac_DA_1_3_std.ply"
idealised_file_bulge = "results/full_stomata_Ac_DA_1_3_bulge.ply"

cs_comparison_output_dir = "results/comparisons_cs" # For 2D side-by-side cross-sections
context_3d_output_dir = "results/comparisons_cs_3d_context" # For 3D context plots
cs_superimposed_output_dir = "results/comparisons_cs_superimposed" # For 2D superimposed cross-sections

if not os.path.exists(cs_comparison_output_dir): os.makedirs(cs_comparison_output_dir)
if not os.path.exists(context_3d_output_dir): os.makedirs(context_3d_output_dir)
if not os.path.exists(cs_superimposed_output_dir): os.makedirs(cs_superimposed_output_dir)


base_name_orig = os.path.splitext(os.path.basename(original_file))[0]

# --- Process Standard Idealised Mesh ---
if os.path.exists(original_file) and os.path.exists(idealised_file_std):
    base_name_vs_std = f"{base_name_orig}_vs_std"
    print(f"\nProcessing Standard Comparison: {base_name_vs_std}")

    (orig_points_2d, orig_aligned_mesh, orig_plane_origin,
     orig_plane_normal, orig_section_verts_3d) = get_midpoint_section_2d_points(original_file)
    (std_points_2d, std_aligned_mesh, std_plane_origin,
     std_plane_normal, std_section_verts_3d) = get_midpoint_section_2d_points(idealised_file_std)

    plot_midpoint_cross_sections_side_by_side(
        original_file, idealised_file_std, cs_comparison_output_dir, base_name_vs_std
    )
    plot_midpoint_cross_sections_superimposed(
        original_file, idealised_file_std, cs_superimposed_output_dir, base_name_vs_std
    )

    if orig_aligned_mesh:
        plot_3d_section_context(orig_aligned_mesh, orig_plane_origin, orig_plane_normal, orig_section_verts_3d,
                                os.path.join(context_3d_output_dir, f"{base_name_orig}_orig_3d_context.png"),
                                mesh_color='grey', section_color='blue')
    if std_aligned_mesh:
        plot_3d_section_context(std_aligned_mesh, std_plane_origin, std_plane_normal, std_section_verts_3d,
                                os.path.join(context_3d_output_dir, f"{base_name_orig}_std_idealised_3d_context.png"),
                                mesh_color='#D55E00', section_color='green')




Processing Standard Comparison: Ac_DA_1_3_vs_std
    Extracting 2D midpoint section for: Meshes/OBJ/Ac_DA_1_3.obj
  Aligning mesh from Meshes/OBJ/Ac_DA_1_3.obj...
  Mesh aligned. Longest axis ([ 0.198  0.98  -0.012]) aligned to Y.
      For midpoint section, using TRANSVERSE plane_origin_3d: [0. 0. 0.] and plane_normal_3d: [0. 1. 0.]
      Fallback: Used longest discrete path for Meshes/OBJ/Ac_DA_1_3.obj.
      Selected polygon with 182 vertices from 1 polygon(s)/path(s) in the 2D section.
      Oriented 2D section horizontally for 2D plots (angle: -79.8°).
      Successfully extracted section for: Meshes/OBJ/Ac_DA_1_3.obj. 2D points for plot: 182, 3D path vertices: 182.
    Extracting 2D midpoint section for: results/full_stomata_Ac_DA_1_3_std.ply
  Aligning mesh from results/full_stomata_Ac_DA_1_3_std.ply...
  Mesh aligned. Longest axis ([ 0.278  0.961 -0.   ]) aligned to Y.
      For midpoint section, using TRANSVERSE plane_origin_3d: [0. 0. 0.] and plane_normal_3d: [0. 1. 0.]
    

ValueError: An even number of points must be given to define each segment.

In [26]:
import trimesh
import pyvista as pv
import numpy as np
import os
from sklearn.decomposition import PCA
import math
# Ensure cross_section_functions is importable, e.g., if it's cross_section_functions.py
# in the same directory or installed.
try:
    import cross_section_functions as csf
except ImportError:
    print("Error: cross_section_functions.py (csf) not found. Please ensure it's accessible.")
    # Fallback DummyCSF (as in your existing code)
    class DummyCSF:
        def load_and_align_mesh(self, mesh_path, align_axis='Y'):
            print(f"DummyCSF: load_and_align_mesh called for {mesh_path}")
            try:
                mesh = trimesh.load_mesh(mesh_path)
                if isinstance(mesh, trimesh.Scene):
                    mesh = mesh.dump(concatenate=True)
                mesh.vertices -= mesh.centroid
                inertia = mesh.moment_inertia_tensor
                eigvals, eigvecs = np.linalg.eigh(inertia)
                longest_axis_vec = eigvecs[:, np.argmax(eigvals)]
                target_axis_vec = np.array([0,1,0] if align_axis == 'Y' else [1,0,0])
                # Simplified rotation logic for dummy
                try:
                    rotation_angle = np.arccos(np.clip(np.dot(longest_axis_vec, target_axis_vec), -1.0, 1.0))
                    rotation_axis = np.cross(longest_axis_vec, target_axis_vec)
                    if np.linalg.norm(rotation_axis) < 1e-6: # Axes are collinear
                        if rotation_angle > np.pi/2: # Axes are opposite
                             rotation_axis = np.array([1,0,0]) if target_axis_vec[0] < 0.9 else np.array([0,0,1]) # pick an arbitrary perp axis
                        else: # Axes are same or nearly same
                             rotation_axis = np.array([0,0,1]) # No rotation needed effectively
                             rotation_angle = 0.0

                    rotation_mat = trimesh.transformations.rotation_matrix(
                        rotation_angle,
                        rotation_axis
                    )[0:3, 0:3]
                    mesh.apply_transform(rotation_mat)
                except Exception as rot_e:
                    print(f"DummyCSF: Error in rotation: {rot_e}")

                return mesh, None 
            except Exception as e:
                print(f"DummyCSF: Error in load_and_align_mesh: {e}")
                return None, None
        
        def get_radial_dimensions(self, aligned_mesh, center, ray_count):
            print(f"DummyCSF: get_radial_dimensions called.")
            min_y, max_y = aligned_mesh.bounds[:,1] if aligned_mesh and aligned_mesh.bounds is not None else (0,1)
            raw_centerline_points = np.array([[0, y, 0] for y in np.linspace(min_y, max_y, 10)])
            minor_radius = np.mean(aligned_mesh.extents)/4 if aligned_mesh and aligned_mesh.extents is not None else 0.5
            return None, None, raw_centerline_points, minor_radius

        def order_points(self, points, method="angular"):
            print(f"DummyCSF: order_points called with {len(points)} points.")
            if len(points) == 0: return np.array([])
            centroid = np.mean(points, axis=0)
            angles = np.arctan2(points[:,1] - centroid[1], points[:,0] - centroid[0])
            return points[np.argsort(angles)]
    
    if 'csf' not in globals(): # If import failed
        csf = DummyCSF()
        print("WARNING: Using DummyCSF as cross_section_functions.py was not found. Results will be incorrect.")


def get_midpoint_section_2d_points(mesh_path):
    """
    Processes a single mesh file to extract its 2D midpoint cross-section (through the wall),
    oriented horizontally for 2D plots. The returned 3D section vertices will correspond to the
    actual cut on the mesh.
    Returns:
        - ordered_points_2d_for_plot_orientation: Horizontally oriented 2D points for 2D plotting.
        - aligned_mesh: The trimesh object of the mesh after alignment.
        - plane_origin_3d: The 3D origin of the cutting plane.
        - plane_normal_3d: The 3D normal of the cutting plane.
        - section_vertices_3d_actual_cut: Ordered 3D vertices of the actual cross-section path on the mesh.
    """
    print(f"    Extracting 2D midpoint section for: {mesh_path}")
    aligned_mesh, _ = csf.load_and_align_mesh(mesh_path, align_axis='Y')
    if aligned_mesh is None:
        print(f"      Failed to load/align mesh: {mesh_path}")
        return None, None, None, None, None

    plane_origin_3d = np.array([0.0, 0.0, 0.0]) 
    plane_normal_3d = np.array([0.0, 1.0, 0.0])
    print(f"      For midpoint section, using TRANSVERSE plane_origin_3d: {plane_origin_3d.round(3)} and plane_normal_3d: {plane_normal_3d.round(3)}")

    section_3D = aligned_mesh.section(plane_origin=plane_origin_3d, plane_normal=plane_normal_3d)

    if section_3D is None or len(section_3D.entities) == 0:
        print(f"      Midpoint section failed or is empty for: {mesh_path}")
        return None, aligned_mesh, plane_origin_3d, plane_normal_3d, None

    path_2D, transform_2d_to_3d = section_3D.to_2D()
    
    raw_polygons_data = path_2D.polygons_closed
    processed_polygons_list = []
    if raw_polygons_data is not None:
        if isinstance(raw_polygons_data, list):
            for item in raw_polygons_data:
                if hasattr(item, 'vertices') and isinstance(item.vertices, np.ndarray):
                    vertices = item.vertices
                    if vertices.ndim == 2 and vertices.shape[1] == 2 and vertices.shape[0] >= 3:
                        processed_polygons_list.append(vertices)
                elif isinstance(item, np.ndarray): 
                    vertices = item
                    if vertices.ndim == 2 and vertices.shape[1] == 2 and vertices.shape[0] >= 3:
                        processed_polygons_list.append(vertices)
        elif isinstance(raw_polygons_data, np.ndarray):
            if raw_polygons_data.ndim == 2 and raw_polygons_data.shape[1] == 2 and raw_polygons_data.shape[0] >= 3:
                processed_polygons_list.append(raw_polygons_data)
            elif raw_polygons_data.ndim > 0 and len(raw_polygons_data) > 0: 
                try:
                    for poly_array in raw_polygons_data:
                        if isinstance(poly_array, np.ndarray) and poly_array.ndim == 2 and poly_array.shape[1] == 2 and poly_array.shape[0] >= 3:
                           processed_polygons_list.append(poly_array)
                except TypeError:
                     print(f"      Warning: polygons_closed (ndarray) has unexpected structure. Mesh: {mesh_path}")
    
    if not processed_polygons_list:
        if hasattr(path_2D, 'discrete') and path_2D.discrete:
            valid_discrete_paths = [p for p in path_2D.discrete if isinstance(p, np.ndarray) and p.ndim == 2 and p.shape[1] == 2 and p.shape[0] >= 3]
            if valid_discrete_paths:
                processed_polygons_list.append(max(valid_discrete_paths, key=len))
                print(f"      Fallback: Used longest discrete path for {mesh_path}.")
            else:
                print(f"      No valid discrete paths found for fallback for: {mesh_path}")
        else:
            print(f"      No closed polygons or discrete paths found for: {mesh_path}")
        if not processed_polygons_list:
            return None, aligned_mesh, plane_origin_3d, plane_normal_3d, None

    final_points_2D_raw_selected_polygon = max(processed_polygons_list, key=len)
    print(f"      Selected polygon with {len(final_points_2D_raw_selected_polygon)} vertices from {len(processed_polygons_list)} polygon(s)/path(s) in the 2D section.")

    if not isinstance(final_points_2D_raw_selected_polygon, np.ndarray):
        final_points_2D_for_ordering = np.array(final_points_2D_raw_selected_polygon)
    else:
        final_points_2D_for_ordering = final_points_2D_raw_selected_polygon

    if len(final_points_2D_for_ordering) < 3:
        print(f"      Not enough points in selected polygon for: {mesh_path} (found {len(final_points_2D_for_ordering)}).")
        return None, aligned_mesh, plane_origin_3d, plane_normal_3d, None

    ordered_points_2d_actual_section = csf.order_points(final_points_2D_for_ordering, method="angular")
    if ordered_points_2d_actual_section is None or len(ordered_points_2d_actual_section) < 3:
        print(f"      Failed to order actual section points for: {mesh_path}")
        return None, aligned_mesh, plane_origin_3d, plane_normal_3d, None

    ordered_points_2d_for_plot_orientation = ordered_points_2d_actual_section 
    if len(ordered_points_2d_actual_section) >= 2:
        try:
            pca = PCA(n_components=2)
            pca.fit(ordered_points_2d_actual_section)
            principal_axis = pca.components_[0] 
            angle_rad = np.arctan2(principal_axis[1], principal_axis[0])
            
            cos_a = np.cos(-angle_rad)
            sin_a = np.sin(-angle_rad)
            rotation_matrix_2d = np.array([[cos_a, -sin_a], [sin_a, cos_a]])
            
            center_2d = np.mean(ordered_points_2d_actual_section, axis=0)
            centered_points_2d = ordered_points_2d_actual_section - center_2d
            rotated_centered_points_2d = centered_points_2d @ rotation_matrix_2d.T
            ordered_points_2d_for_plot_orientation = rotated_centered_points_2d + center_2d
            print(f"      Oriented 2D section horizontally for 2D plots (angle: {-np.degrees(angle_rad):.1f}°).")
        except Exception as e:
            print(f"      Warning: PCA-based orientation for 2D plots failed: {e}")

    section_vertices_3d_actual_cut = None
    if ordered_points_2d_actual_section is not None and len(ordered_points_2d_actual_section) > 0:
        if transform_2d_to_3d is not None:
            current_2d_points_for_3d = ordered_points_2d_actual_section
            if current_2d_points_for_3d.ndim == 1:
                 current_2d_points_for_3d = current_2d_points_for_3d.reshape(-1, 2)
            points_in_2d_plane_for_transform = np.column_stack((current_2d_points_for_3d, np.zeros(len(current_2d_points_for_3d))))
            section_vertices_3d_actual_cut = trimesh.transform_points(points_in_2d_plane_for_transform, transform_2d_to_3d)
        else:
            print(f"      Warning: transform_2d_to_3d is None for {mesh_path}. Attempting to use 3D polygons/paths directly from section_3D.")
            if hasattr(section_3D, 'polygons_closed') and section_3D.polygons_closed:
                polygons_3d_list = [p for p in section_3D.polygons_closed if isinstance(p, np.ndarray) and p.ndim == 2 and p.shape[0] >=3]
                if polygons_3d_list:
                    section_vertices_3d_actual_cut = max(polygons_3d_list, key=len)
                    print(f"      Using largest 3D polygon from section_3D.polygons_closed (length {len(section_vertices_3d_actual_cut)}) as fallback for 3D section vertices.")
            if section_vertices_3d_actual_cut is None and hasattr(section_3D, 'discrete') and section_3D.discrete:
                 polylines_3d_list = [p for p in section_3D.discrete if isinstance(p, np.ndarray) and p.ndim == 2 and p.shape[0] >=3]
                 if polylines_3d_list:
                    section_vertices_3d_actual_cut = max(polylines_3d_list, key=len)
                    print(f"      Using largest discrete 3D polyline from section_3D.discrete (length {len(section_vertices_3d_actual_cut)}) as fallback for 3D section vertices.")
            if section_vertices_3d_actual_cut is None:
                 print(f"      No suitable 3D path fallback found in section_3D when transform_2d_to_3d is None.")

    if ordered_points_2d_for_plot_orientation is None or len(ordered_points_2d_for_plot_orientation) == 0:
        print(f"      Final ordered_points_2d_for_plot_orientation is empty for: {mesh_path}")
        return None, aligned_mesh, plane_origin_3d, plane_normal_3d, None

    if section_vertices_3d_actual_cut is None or len(section_vertices_3d_actual_cut) == 0:
        print(f"      Final section_vertices_3d_actual_cut is empty for: {mesh_path}")
        return ordered_points_2d_for_plot_orientation, aligned_mesh, plane_origin_3d, plane_normal_3d, None

    print(f"      Successfully extracted section for: {mesh_path}. 2D points for plot: {len(ordered_points_2d_for_plot_orientation)}, 3D path vertices: {len(section_vertices_3d_actual_cut)}.")
    return ordered_points_2d_for_plot_orientation, aligned_mesh, plane_origin_3d, plane_normal_3d, section_vertices_3d_actual_cut

# --- ADD THIS FUNCTION DEFINITION ---
def plot_midpoint_cross_sections_side_by_side(original_mesh_path, idealised_mesh_path, output_dir, base_name):
    """
    Generates a side-by-side plot of the 2D midpoint cross-section outlines of two meshes.
    """
    print(f"\nPlotting side-by-side midpoint cross-section outlines for '{base_name}'")
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    points_orig_2d, _, _, _, _ = get_midpoint_section_2d_points(original_mesh_path)
    points_ideal_2d, _, _, _, _ = get_midpoint_section_2d_points(idealised_mesh_path)

    if points_orig_2d is None and points_ideal_2d is None:
        print("  Both mesh cross-sections failed. Skipping side-by-side plot.")
        return

    plotter = pv.Plotter(shape=(1, 2), off_screen=True, window_size=[2048, 1024], border=False)

    original_color = 'grey'
    idealised_color = '#D55E00' 
    line_width = 5
    ruler_length_world = 5.0
    
    all_plot_points = []
    if points_orig_2d is not None and len(points_orig_2d) > 0:
        all_plot_points.extend(points_orig_2d)
    if points_ideal_2d is not None and len(points_ideal_2d) > 0:
        all_plot_points.extend(points_ideal_2d)

    global_min_x, global_max_x, global_min_y, global_max_y = (np.inf, -np.inf, np.inf, -np.inf)
    has_data_for_bounds = False

    if all_plot_points:
        all_points_np = np.array(all_plot_points)
        if all_points_np.size > 0 and all_points_np.ndim == 2 and all_points_np.shape[1] == 2:
            global_min_x = np.min(all_points_np[:,0])
            global_max_x = np.max(all_points_np[:,0])
            global_min_y = np.min(all_points_np[:,1])
            global_max_y = np.max(all_points_np[:,1])
            has_data_for_bounds = True
        else:
            print("     Warning: all_plot_points is not in the expected format for bounds calculation (side-by-side).")
            
    padding_for_ruler_area_y = ruler_length_world * 0.4 

    if has_data_for_bounds:
        global_min_y_padded = global_min_y - padding_for_ruler_area_y
        min_extent = ruler_length_world * 0.5 
        if (global_max_x - global_min_x) < min_extent:
            center_x_bounds = (global_min_x + global_max_x) / 2
            global_min_x, global_max_x = center_x_bounds - min_extent / 2, center_x_bounds + min_extent / 2
        if (global_max_y - global_min_y) < min_extent: 
            center_y_bounds = (global_min_y + global_max_y) / 2
            global_min_y, global_max_y = center_y_bounds - min_extent / 2, center_y_bounds + min_extent / 2
            global_min_y_padded = global_min_y - padding_for_ruler_area_y 
    else: 
        global_min_x, global_max_x = -ruler_length_world, ruler_length_world
        global_min_y_padded = -ruler_length_world * 0.7 
        global_max_y = ruler_length_world * 0.3

    text_viewport_pos_subplot = (0.47, 0.05) 

    # --- Subplot 1: Original Mesh Cross-Section ---
    plotter.subplot(0, 0)
    # Removed justification='center'
    plotter.add_text("Original Mesh Cross-section", position=(0.5, 0.9), viewport=True, font_size=12, color='black', shadow=True, name="title_s1", font='arial')
    if points_orig_2d is not None and len(points_orig_2d) > 1:
        points_orig_3d = np.hstack((points_orig_2d, np.zeros((points_orig_2d.shape[0], 1))))
        closed_points_orig = np.vstack([points_orig_3d, points_orig_3d[0]])
        plotter.add_lines(closed_points_orig, color=original_color, width=line_width, connected=True)
    
    if has_data_for_bounds or (points_orig_2d is not None):
        ruler_center_x_s1 = (global_min_x + global_max_x) / 2.0
        ruler_y_pos_s1 = global_min_y_padded + (padding_for_ruler_area_y * 0.25)
        p0_ruler_s1 = np.array([ruler_center_x_s1 - ruler_length_world / 2.0, ruler_y_pos_s1, 0])
        p1_ruler_s1 = np.array([ruler_center_x_s1 + ruler_length_world / 2.0, ruler_y_pos_s1, 0])
        plotter.add_mesh(pv.Line(p0_ruler_s1, p1_ruler_s1), color='black', line_width=line_width)
        plotter.add_text("5 µm", position=text_viewport_pos_subplot, font_size=16, color='black', font='arial', viewport=True, shadow=True)

    plotter.view_xy()
    plotter.enable_parallel_projection()
    plotter.camera.bounds = [global_min_x, global_max_x, global_min_y_padded, global_max_y, -1, 1]
    plotter.reset_camera()
    plotter.camera.zoom(1.1)

    # --- Subplot 2: Idealised Mesh Cross-Section ---
    plotter.subplot(0, 1)
    # Removed justification='center'
    plotter.add_text("Idealised Mesh Cross-section", position=(0.5, 0.9), viewport=True, font_size=12, color='black', shadow=True, name="title_s2", font='arial')
    if points_ideal_2d is not None and len(points_ideal_2d) > 1:
        points_ideal_3d = np.hstack((points_ideal_2d, np.zeros((points_ideal_2d.shape[0], 1))))
        closed_points_ideal = np.vstack([points_ideal_3d, points_ideal_3d[0]])
        plotter.add_lines(closed_points_ideal, color=idealised_color, width=line_width, connected=True)

    if has_data_for_bounds or (points_ideal_2d is not None): 
        ruler_center_x_s2 = (global_min_x + global_max_x) / 2.0 
        ruler_y_pos_s2 = global_min_y_padded + (padding_for_ruler_area_y * 0.25)
        p0_ruler_s2 = np.array([ruler_center_x_s2 - ruler_length_world / 2.0, ruler_y_pos_s2, 0])
        p1_ruler_s2 = np.array([ruler_center_x_s2 + ruler_length_world / 2.0, ruler_y_pos_s2, 0])
        plotter.add_mesh(pv.Line(p0_ruler_s2, p1_ruler_s2), color='black', line_width=line_width)
        plotter.add_text("5 µm", position=text_viewport_pos_subplot, font_size=16, color='black', font='arial', viewport=True, shadow=True)

    plotter.view_xy()
    plotter.enable_parallel_projection()
    plotter.camera.bounds = [global_min_x, global_max_x, global_min_y_padded, global_max_y, -1, 1]
    plotter.reset_camera()
    plotter.camera.zoom(1.1)

    plotter.link_views() 

    output_filename = f"{base_name}_midpoint_cs_sidebyside.png"
    output_path = os.path.join(output_dir, output_filename)
    print(f"  Saving side-by-side 2D cross-section to: {output_path}")
    plotter.screenshot(output_path, transparent_background=True)
    plotter.close()

# --- END OF ADDED FUNCTION ---

def plot_3d_section_context(trimesh_mesh_aligned, plane_origin, plane_normal, section_path_vertices_3d, output_filename, mesh_color='lightgrey', section_color='red', plane_color='blue'):
    """
    Generates a 3D visualization of the mesh with its cutting plane and section path.
    Also generates a top-down view showing the section position.
    """
    if trimesh_mesh_aligned is None:
        print(f"  Skipping 3D context plot for {output_filename}, mesh is None.")
        return

    pv_mesh = pv.wrap(trimesh_mesh_aligned)

    plotter_iso = pv.Plotter(off_screen=True, window_size=[1280, 960], border=False)
    plotter_iso.add_mesh(pv_mesh, color=mesh_color, opacity=0.7, smooth_shading=False, show_edges=False)

    if plane_origin is not None and plane_normal is not None:
        if section_path_vertices_3d is not None and len(section_path_vertices_3d) > 1:
            closed_section_path_3d = np.vstack([section_path_vertices_3d, section_path_vertices_3d[0]])
            plotter_iso.add_lines(closed_section_path_3d, color=section_color, width=10, connected=True)

        plotter_iso.view_isometric()
        plotter_iso.enable_parallel_projection()
        plotter_iso.reset_camera()
        plotter_iso.camera.zoom(1.5)
    else:
        plotter_iso.view_isometric()
        plotter_iso.enable_parallel_projection()
        plotter_iso.reset_camera()
        plotter_iso.camera.zoom(1.3)

    print(f"  Saving 3D isometric section context view to: {output_filename}")
    plotter_iso.screenshot(output_filename, transparent_background=True)
    plotter_iso.close()

    if plane_origin is not None and plane_normal is not None and section_path_vertices_3d is not None and len(section_path_vertices_3d) > 1:
        plotter_top = pv.Plotter(off_screen=True, window_size=[1024, 1024], border=False)
        plotter_top.add_mesh(pv_mesh, color=mesh_color, opacity=0.85, smooth_shading=False, show_edges=False)
        
        closed_section_path_3d_top = np.vstack([section_path_vertices_3d, section_path_vertices_3d[0]])
        plotter_top.add_lines(closed_section_path_3d_top, color=section_color, width=10, connected=True)

        plotter_top.view_xy() 
        plotter_top.enable_parallel_projection()
        plotter_top.reset_camera() 
        plotter_top.camera.zoom(1.4) 

        base, ext = os.path.splitext(output_filename)
        top_view_output_filename = f"{base}_top_view{ext}"
        print(f"  Saving 3D top-down section context view to: {top_view_output_filename}")
        plotter_top.screenshot(top_view_output_filename, transparent_background=True)
        plotter_top.close()
    elif plane_origin is None or plane_normal is None:
        print(f"  Skipping top-down view for {output_filename} as plane information is missing.")
    elif section_path_vertices_3d is None or len(section_path_vertices_3d) <= 1:
        print(f"  Skipping top-down view for {output_filename} as section path is missing or too short.")


def plot_midpoint_cross_sections_superimposed(original_mesh_path, idealised_mesh_path, output_dir, base_name):
    """
    Generates a superimposed plot of the 2D midpoint cross-section outlines of two meshes.
    """
    print(f"\nPlotting superimposed midpoint cross-section outlines for '{base_name}'")
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    points_orig_2d, _, _, _, _ = get_midpoint_section_2d_points(original_mesh_path)
    points_ideal_2d, _, _, _, _ = get_midpoint_section_2d_points(idealised_mesh_path)

    if points_orig_2d is None and points_ideal_2d is None:
        print("  Both mesh cross-sections failed. Skipping superimposed plot.")
        return

    plotter = pv.Plotter(off_screen=True, window_size=[1024, 1024], border=False) 

    original_color = 'grey' 
    idealised_color = '#D55E00' 
    line_width = 5
    ruler_length_world = 5.0
    text_viewport_pos = (0.47, 0.05) 

    all_plot_points = []
    if points_orig_2d is not None and len(points_orig_2d) > 0:
        all_plot_points.extend(points_orig_2d)
    if points_ideal_2d is not None and len(points_ideal_2d) > 0:
        all_plot_points.extend(points_ideal_2d)

    global_min_x, global_max_x, global_min_y, global_max_y = (np.inf, -np.inf, np.inf, -np.inf)
    has_data_for_bounds = False

    if all_plot_points:
        all_points_np = np.array(all_plot_points)
        if all_points_np.size > 0 and all_points_np.ndim == 2 and all_points_np.shape[1] == 2:
            global_min_x = np.min(all_points_np[:,0])
            global_max_x = np.max(all_points_np[:,0])
            global_min_y = np.min(all_points_np[:,1])
            global_max_y = np.max(all_points_np[:,1])
            has_data_for_bounds = True
        else:
            print("     Warning: all_plot_points is not in the expected format for bounds calculation (superimposed).")

    padding_for_ruler_area_y = ruler_length_world * 0.4

    if has_data_for_bounds:
        global_min_y_padded = global_min_y - padding_for_ruler_area_y
        min_extent = ruler_length_world * 0.5
        if (global_max_x - global_min_x) < min_extent:
            center_x = (global_min_x + global_max_x) / 2
            global_min_x, global_max_x = center_x - min_extent / 2, center_x + min_extent / 2
        if (global_max_y - global_min_y) < min_extent:
            center_y = (global_min_y + global_max_y) / 2
            global_min_y, global_max_y = center_y - min_extent / 2, center_y + min_extent / 2
            global_min_y_padded = global_min_y - padding_for_ruler_area_y
    else:
        global_min_x, global_max_x = -ruler_length_world, ruler_length_world
        global_min_y_padded = -ruler_length_world * 0.7
        global_max_y = ruler_length_world * 0.3

    legend_entries = []
    if points_orig_2d is not None and len(points_orig_2d) > 1:
        points_orig_3d = np.hstack((points_orig_2d, np.zeros((points_orig_2d.shape[0], 1))))
        closed_points_orig = np.vstack([points_orig_3d, points_orig_3d[0]])
        plotter.add_lines(closed_points_orig, color=original_color, width=line_width, connected=True)        
        legend_entries.append(['Original', original_color])

    if points_ideal_2d is not None and len(points_ideal_2d) > 1:
        points_ideal_3d = np.hstack((points_ideal_2d, np.zeros((points_ideal_2d.shape[0], 1))))
        closed_points_ideal = np.vstack([points_ideal_3d, points_ideal_3d[0]])
        # Corrected color for idealised mesh in superimposed plot
        plotter.add_lines(closed_points_ideal, color=idealised_color, width=line_width, connected=True)        
        legend_entries.append(['Idealised', idealised_color])

    if has_data_for_bounds or legend_entries: 
        ruler_center_x = (global_min_x + global_max_x) / 2.0
        ruler_y_pos = global_min_y_padded + (padding_for_ruler_area_y * 0.25)
        p0_ruler = np.array([ruler_center_x - ruler_length_world / 2.0, ruler_y_pos, 0])
        p1_ruler = np.array([ruler_center_x + ruler_length_world / 2.0, ruler_y_pos, 0])
        plotter.add_mesh(pv.Line(p0_ruler, p1_ruler), color='black', line_width=line_width) # Use consistent line_width
        plotter.add_text("5 µm", position=text_viewport_pos, font_size=16, color='black', font='arial', viewport=True, shadow=True)

    if legend_entries:
        plotter.add_legend(labels=legend_entries, bcolor=None, face=None, size=(0.15, 0.15))


    plotter.view_xy()
    plotter.enable_parallel_projection()
    plotter.camera.bounds = [global_min_x, global_max_x, global_min_y_padded, global_max_y, -1, 1]
    plotter.reset_camera()
    plotter.camera.zoom(1.1) # Slight zoom

    output_filename = f"{base_name}_midpoint_cs_superimposed.png"
    output_path = os.path.join(output_dir, output_filename)
    print(f"  Saving superimposed 2D cross-section to: {output_path}")
    plotter.screenshot(output_path, transparent_background=True)
    plotter.close()

In [ ]:
# --- Process Idealised Mesh ---
if os.path.exists(original_file) and os.path.exists(idealised_file_std): # Check for idealised_file_std
    base_name_vs_std_pass2 = f"{base_name_orig}_vs_std_pass2" # Unique basename for this pass
    print(f"\nProcessing Standard Comparison (Second Pass): {base_name_vs_std_pass2}")

    # Re-fetch original data. Variables are suffixed with _b to distinguish from any prior original data.
    (orig_points_2d_b, orig_aligned_mesh_b, orig_plane_origin_b,
     orig_plane_normal_b, orig_section_verts_3d_b) = get_midpoint_section_2d_points(original_file)
    
    # Get data for the standard idealised mesh. Variables suffixed with _std_p2 for "standard pass 2".
    (std_p2_points_2d, std_p2_aligned_mesh, std_p2_plane_origin,
     std_p2_plane_normal, std_p2_section_verts_3d) = get_midpoint_section_2d_points(idealised_file_std)

    plot_midpoint_cross_sections_side_by_side(
        original_file, idealised_file_std, cs_comparison_output_dir, base_name_vs_std_pass2
    )
    plot_midpoint_cross_sections_superimposed(
        original_file, idealised_file_std, cs_superimposed_output_dir, base_name_vs_std_pass2
    )

    # Plot 3D context for original mesh (for this comparison pass)
    if orig_aligned_mesh_b:
        plot_3d_section_context(orig_aligned_mesh_b, orig_plane_origin_b, orig_plane_normal_b, orig_section_verts_3d_b,
                                os.path.join(context_3d_output_dir, f"{base_name_orig}_orig_for_std_pass2_3d_context.png"),
                                mesh_color='grey', section_color='blue') # Standard colors for original
    
    # Plot 3D context for the standard idealised mesh (for this comparison pass)
    if std_p2_aligned_mesh:
        plot_3d_section_context(std_p2_aligned_mesh, std_p2_plane_origin, std_p2_plane_normal, std_p2_section_verts_3d,
                                os.path.join(context_3d_output_dir, f"{base_name_orig}_std_idealised_pass2_3d_context.png"),
                                mesh_color='#D55E00', section_color='green') # Standard colors for std idealised

print("\nProcessing complete.")


Processing Bulge Comparison: Ac_DA_1_3_vs_bulge
    Extracting 2D midpoint section for: Meshes/OBJ/Ac_DA_1_3.obj
  Aligning mesh from Meshes/OBJ/Ac_DA_1_3.obj...
  Mesh aligned. Longest axis ([ 0.198  0.98  -0.012]) aligned to Y.
      For midpoint section, using TRANSVERSE plane_origin_3d: [0. 0. 0.] and plane_normal_3d: [0. 1. 0.]
      Fallback: Used longest discrete path for Meshes/OBJ/Ac_DA_1_3.obj.
      Selected polygon with 182 vertices from 1 polygon(s)/path(s) in the 2D section.
      Oriented 2D section horizontally for 2D plots (angle: -79.8°).
      Successfully extracted section for: Meshes/OBJ/Ac_DA_1_3.obj. 2D points for plot: 182, 3D path vertices: 182.
    Extracting 2D midpoint section for: results/full_stomata_Ac_DA_1_3_bulge.ply
  Aligning mesh from results/full_stomata_Ac_DA_1_3_bulge.ply...
  Mesh aligned. Longest axis ([ 0.278  0.961 -0.   ]) aligned to Y.
      For midpoint section, using TRANSVERSE plane_origin_3d: [0. 0. 0.] and plane_normal_3d: [0. 1. 0.]
 